# 第五章：高级检索策略

## 学习目标
- 理解基础向量检索的局限性
- 实现 BM25 关键词检索
- 构建混合检索（向量 + BM25）
- 使用节点后处理器过滤和排序结果
- 掌握 HyDE 查询变换技术
- 使用 SubQuestionQueryEngine 分解复杂问题

## 0. 环境准备

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

from llama_index.llms.openai_like import OpenAILike
from llama_index.embeddings.openai_like import OpenAILikeEmbedding
from llama_index.core import Settings

# 初始化 LLM（通过 OpenAI 兼容接口接入 Qwen）
llm = OpenAILike(
    model="qwen-plus",
    api_base=os.environ["Qwen_API_BASE"],
    api_key=os.environ["Qwen_API_KEY"],
    is_chat_model=True,
)
Settings.llm = llm

# 切换为 GLM（同样是 OpenAI 兼容接口）：
# from llama_index.llms.openai_like import OpenAILike
# Settings.llm = OpenAILike(model="glm-4-plus", api_base=os.environ["GLM_API_BASE"], api_key=os.environ["GLM_API_KEY"], is_chat_model=True)

# 初始化 Embedding 模型
Settings.embed_model = OpenAILikeEmbedding(
    model_name="text-embedding-v3",
    api_base=os.environ["Qwen_API_BASE"],
    api_key=os.environ["Qwen_API_KEY"],
)

print("环境初始化完成")

### 构建知识库

为了让不同检索策略的差异更明显，我们准备一批涵盖多种编程主题的文档。每篇文档包含 2-3 句话，话题之间有一定重叠但侧重点不同。

In [ ]:
from llama_index.core import Document

documents = [
    Document(
        text="Python 是一种解释型、面向对象的高级编程语言，语法简洁易读。Python 广泛应用于 Web 开发、数据分析和自动化脚本编写，拥有丰富的标准库和第三方生态。",
        metadata={"topic": "Python 基础"},
    ),
    Document(
        text="Python 的高级特性包括装饰器、生成器、上下文管理器和元类。装饰器用 @decorator 语法在不修改函数源码的前提下增强功能，生成器通过 yield 实现惰性求值，极大节省内存开销。",
        metadata={"topic": "Python 高级特性"},
    ),
    Document(
        text="JavaScript 是 Web 前端的核心语言，运行在浏览器环境中。ES6 引入了箭头函数、Promise、async/await 等现代特性，使异步编程更加优雅。Node.js 让 JavaScript 也能用于服务端开发。",
        metadata={"topic": "JavaScript 基础"},
    ),
    Document(
        text="Web 开发通常分为前端和后端。前端使用 HTML、CSS、JavaScript 构建用户界面，后端使用 Python、Java、Go 等语言处理业务逻辑和数据存储。RESTful API 是前后端通信的主流方式。",
        metadata={"topic": "Web 开发"},
    ),
    Document(
        text="关系型数据库（如 MySQL、PostgreSQL）使用 SQL 语言进行数据操作，支持事务和 ACID 特性。NoSQL 数据库（如 MongoDB、Redis）则适合非结构化数据和高并发场景，牺牲部分一致性换取性能。",
        metadata={"topic": "数据库概念"},
    ),
    Document(
        text="API 设计的核心原则包括一致性、版本控制和错误处理。RESTful API 使用 HTTP 动词（GET、POST、PUT、DELETE）对资源进行操作，GraphQL 则允许客户端精确指定需要的数据字段，减少过度获取。",
        metadata={"topic": "API 设计"},
    ),
    Document(
        text="测试策略包括单元测试、集成测试和端到端测试。单元测试验证单个函数或方法的正确性，使用 pytest 或 unittest 框架。测试驱动开发（TDD）要求先写测试再写实现，有助于提高代码质量和可维护性。",
        metadata={"topic": "测试策略"},
    ),
    Document(
        text="CI/CD（持续集成/持续部署）是 DevOps 的核心实践。CI 指开发者频繁将代码合并到主干并自动运行测试，CD 指通过自动化流水线将通过测试的代码部署到生产环境。常用工具有 Jenkins、GitHub Actions 和 GitLab CI。",
        metadata={"topic": "DevOps/CI-CD"},
    ),
]

print(f"共创建 {len(documents)} 个文档：")
for doc in documents:
    print(f"  [{doc.metadata['topic']}] {doc.text[:40]}...")

In [ ]:
from llama_index.core import VectorStoreIndex

# 构建向量索引
index = VectorStoreIndex.from_documents(documents)
print("向量索引构建完成")

## 1. 基础向量检索的局限

在前几章中我们一直使用向量检索——将问题和文档都转为 embedding 向量，用余弦相似度找最相关的文本块。这种方式在大多数情况下效果不错，但它有一个天然弱点：**对精确关键词匹配不敏感**。

In [ ]:
# 基础向量检索
query_engine = index.as_query_engine(similarity_top_k=2)
response = query_engine.query("CI/CD 是什么？")
print("回答:", response)
print()
print("检索到的节点:")
for node in response.source_nodes:
    print(f"  [{node.metadata['topic']}] 相似度: {node.score:.4f}")
    print(f"  {node.text[:60]}...")
    print()

In [ ]:
# 试一个更挑战的查询：包含特定技术术语
response2 = query_engine.query("pytest 框架怎么用？")
print("回答:", response2)
print()
print("检索到的节点:")
for node in response2.source_nodes:
    print(f"  [{node.metadata['topic']}] 相似度: {node.score:.4f}")
    print(f"  {node.text[:60]}...")
    print()

### 刚才发生了什么？

向量检索的工作原理是计算语义相似度。当查询和文档在**语义空间**中靠近时，能匹配到。但以下场景可能出现问题：

| 场景 | 问题 |
|------|------|
| 搜索特定术语（如 `pytest`、`Jenkins`） | 向量可能捕捉到更宽泛的「测试」语义，但遗漏精确关键词 |
| 搜索缩写或专有名词（如 `CI/CD`、`TDD`） | 语义模型可能不熟悉某些缩写，匹配不精确 |
| 查询很短（1-2 个词） | embedding 的信息量不足，容易匹配到无关内容 |
| 文档中关键信息只出现一次 | 向量可能被周围的语义「淹没」 |

这就是为什么我们需要更高级的检索策略——不能只依赖向量检索。

## 2. BM25 关键词检索

BM25 是经典的信息检索算法（搜索引擎 Elasticsearch 默认使用的算法），基于**词频**（TF）和**逆文档频率**（IDF）来计算相关性。简单来说：一个词在某文档中出现得多、在其他文档中出现得少，那它对这个文档的重要性就高。

In [ ]:
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.node_parser import SentenceSplitter

# 先将文档切分为节点（BM25 检索器需要节点列表）
splitter = SentenceSplitter(chunk_size=256)
nodes = splitter.get_nodes_from_documents(documents)

print(f"切分为 {len(nodes)} 个节点")

In [ ]:
# BM25 检索器（基于关键词匹配）
bm25_retriever = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=3)
results = bm25_retriever.retrieve("CI/CD 持续集成")

print("BM25 检索结果:")
for node in results:
    print(f"  [{node.metadata.get('topic', '?')}] 分数: {node.score:.4f}")
    print(f"  {node.text[:60]}...")
    print()

In [ ]:
# 对比：用 BM25 搜索 "pytest"
results_bm25 = bm25_retriever.retrieve("pytest 框架")

print("BM25 搜索 'pytest 框架':")
for node in results_bm25:
    print(f"  [{node.metadata.get('topic', '?')}] 分数: {node.score:.4f}")
    print(f"  {node.text[:60]}...")
    print()

### 刚才发生了什么？

BM25 检索器直接基于**关键词匹配**来打分。当文档中包含查询中的精确词汇时，BM25 能精准命中。

BM25 的核心思想：

$$\text{BM25}(q, d) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{\text{tf}(t, d) \cdot (k_1 + 1)}{\text{tf}(t, d) + k_1 \cdot (1 - b + b \cdot \frac{|d|}{\text{avgdl}})}$$

不用理解公式细节，只需记住：
- **词频（TF）**：这个词在文档中出现越多，得分越高
- **逆文档频率（IDF）**：这个词越稀有（只在少数文档中出现），得分越高
- **文档长度归一化**：短文档中出现关键词，比长文档中更有意义

BM25 的优势和劣势：

| 优势 | 劣势 |
|------|------|
| 精确关键词匹配，命中率高 | 无法理解同义词（搜「测试」找不到「验证」） |
| 不需要 GPU 或 Embedding API | 无法理解语义（搜「保证代码质量」找不到测试相关文档） |
| 速度极快 | 对中文分词质量有依赖 |

## 3. 混合检索（向量 + BM25）

既然向量检索擅长语义匹配、BM25 擅长关键词匹配，那把它们结合起来就是最佳策略——这就是**混合检索（Hybrid Retrieval）**。

LlamaIndex 提供了 `QueryFusionRetriever`，内置了 **RRF（Reciprocal Rank Fusion）** 融合算法，能将多个检索器的结果合并排序。

In [ ]:
# ❌ 纯向量检索
vector_retriever = index.as_retriever(similarity_top_k=3)
vector_results = vector_retriever.retrieve("CI/CD 持续集成")

print("❌ 纯向量检索结果:")
for node in vector_results:
    print(f"  [{node.metadata.get('topic', '?')}] 分数: {node.score:.4f}")
print()

In [ ]:
# ❌ 纯 BM25 检索
bm25_results = bm25_retriever.retrieve("CI/CD 持续集成")

print("❌ 纯 BM25 检索结果:")
for node in bm25_results:
    print(f"  [{node.metadata.get('topic', '?')}] 分数: {node.score:.4f}")
print()

In [ ]:
# ✅ 混合检索（向量 + BM25 + RRF 融合）
from llama_index.core.retrievers import QueryFusionRetriever

hybrid_retriever = QueryFusionRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    similarity_top_k=3,
    num_queries=1,       # 不生成额外查询变体
    mode="reciprocal_rerank",  # 使用 RRF 融合排序
)

hybrid_results = hybrid_retriever.retrieve("CI/CD 持续集成")

print("✅ 混合检索结果:")
for node in hybrid_results:
    print(f"  [{node.metadata.get('topic', '?')}] 分数: {node.score:.4f}")
    print(f"  {node.text[:60]}...")
    print()

### 刚才发生了什么？

`QueryFusionRetriever` 将多个检索器的结果融合为一个排序列表。融合过程：

1. 分别调用向量检索器和 BM25 检索器
2. 收集所有结果及其排名
3. 使用 **RRF（Reciprocal Rank Fusion）** 算法计算最终分数：

$$\text{RRF}(d) = \sum_{r \in R} \frac{1}{k + \text{rank}_r(d)}$$

其中 $k$ 是一个常数（默认 60），$\text{rank}_r(d)$ 是文档在第 $r$ 个检索器中的排名。RRF 的巧妙之处在于：一个文档只要在**任意一个检索器**中排名靠前，它的最终得分就会高。

关键参数说明：

| 参数 | 说明 |
|------|------|
| `retrievers` | 要融合的检索器列表 |
| `similarity_top_k` | 最终返回的节点数量 |
| `num_queries` | 自动生成的查询变体数量（1 表示不生成额外变体，只用原始查询） |
| `mode` | 融合策略，`reciprocal_rerank` 是最常用的 RRF |

### 三种检索策略对比

| 检索策略 | 优势 | 劣势 |
|---------|------|------|
| 向量检索 | 捕捉语义相似性，理解同义词 | 可能遗漏精确关键词 |
| BM25 | 精确关键词匹配，速度快 | 无法理解同义词和语义 |
| 混合检索 | 两者优势互补，覆盖面广 | 需要调优融合权重 |

> **与 LangChain 对比**：LangChain 也有 `EnsembleRetriever` 实现混合检索，但需要手动安装 `rank_bm25` 包，并自行封装 BM25 检索器。LlamaIndex 的 `QueryFusionRetriever` 内置了 RRF 融合，开箱即用，配置更简洁。

## 4. 节点后处理器

检索只是第一步——拿到候选节点后，往往还需要**过滤**或**重排序**。LlamaIndex 提供了「节点后处理器（Node Postprocessor）」机制，它们位于检索器和响应合成器之间，就像一条流水线上的质检环节。

In [ ]:
from llama_index.core.postprocessor import SimilarityPostprocessor

# 相似度阈值过滤：丢弃相似度低于 0.5 的节点
similarity_filter = SimilarityPostprocessor(similarity_cutoff=0.5)

# 先看一下不过滤时的结果
retriever = index.as_retriever(similarity_top_k=5)
raw_results = retriever.retrieve("Python 有哪些高级特性？")

print("过滤前（top-5）:")
for node in raw_results:
    print(f"  [{node.metadata['topic']}] 相似度: {node.score:.4f}")

In [ ]:
# 应用相似度过滤
filtered_results = similarity_filter.postprocess_nodes(raw_results)

print("过滤后（相似度 >= 0.5）:")
for node in filtered_results:
    print(f"  [{node.metadata['topic']}] 相似度: {node.score:.4f}")

print(f"\n过滤掉了 {len(raw_results) - len(filtered_results)} 个低相关节点")

In [ ]:
from llama_index.core.postprocessor import KeywordNodePostprocessor

# 关键词过滤：只保留包含 "Python" 的节点
keyword_filter = KeywordNodePostprocessor(
    required_keywords=["Python"],
)

keyword_filtered = keyword_filter.postprocess_nodes(raw_results)

print("关键词过滤后（包含 'Python'）:")
for node in keyword_filtered:
    print(f"  [{node.metadata['topic']}] {node.text[:50]}...")

In [ ]:
# 在查询引擎中集成后处理器
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.response_synthesizers import get_response_synthesizer

engine = RetrieverQueryEngine(
    retriever=index.as_retriever(similarity_top_k=5),
    node_postprocessors=[similarity_filter],
    response_synthesizer=get_response_synthesizer(),
)

response = engine.query("Python 有哪些高级特性？")
print(response)
print(f"\n过滤后使用了 {len(response.source_nodes)} 个节点")

### 刚才发生了什么？

节点后处理器在**检索之后、合成回答之前**对节点进行处理。LlamaIndex 内置了多种后处理器：

| 后处理器 | 功能 |
|---------|------|
| `SimilarityPostprocessor` | 按相似度阈值过滤，丢弃低分节点 |
| `KeywordNodePostprocessor` | 按关键词过滤，要求节点包含/排除特定关键词 |
| `MetadataReplacementPostProcessor` | 用元数据中的特定字段替换节点文本 |
| `LongContextReorder` | 重排序节点，将最相关的放在开头和结尾（利用 LLM 对首尾信息更敏感的特点） |

后处理器可以叠加使用——传入列表即可依次执行：

```python
node_postprocessors=[similarity_filter, keyword_filter]
```

数据流：**Retriever → Postprocessor 1 → Postprocessor 2 → ... → Response Synthesizer**

> **与 LangChain 对比**：LangChain 没有内置的后处理器机制。要实现类似功能，你需要自定义过滤逻辑，通常写在 Retriever 和 Chain 之间。LlamaIndex 将这一层标准化了，用声明式配置代替了手写过滤代码。

## 5. HyDE 查询变换

**HyDE（Hypothetical Document Embeddings）** 是一种巧妙的检索优化技术。核心思路：

1. 用户提出一个问题
2. 让 LLM 先生成一个**假设性答案**（即使不完全准确）
3. 用这个假设性答案的 embedding 去检索（而不是用原始问题）

为什么这样做更好？因为假设性答案在语义空间中通常比原始问题**更接近**真正的答案文档。

In [ ]:
# ❌ 普通查询：用问题本身的 embedding 检索
base_engine = index.as_query_engine(similarity_top_k=3)
response_base = base_engine.query("如何保证代码质量？")

print("❌ 普通查询:")
print("回答:", response_base)
print("\n检索到的节点:")
for node in response_base.source_nodes:
    print(f"  [{node.metadata['topic']}] 相似度: {node.score:.4f}")

In [ ]:
# ✅ HyDE 查询变换：先生成假设性答案，再用答案的 embedding 检索
from llama_index.core.indices.query.query_transform import HyDEQueryTransform
from llama_index.core.query_engine import TransformQueryEngine

hyde = HyDEQueryTransform(include_original=True)
hyde_engine = TransformQueryEngine(base_engine, query_transform=hyde)

response_hyde = hyde_engine.query("如何保证代码质量？")

print("✅ HyDE 查询:")
print("回答:", response_hyde)
print("\n检索到的节点:")
for node in response_hyde.source_nodes:
    print(f"  [{node.metadata['topic']}] 相似度: {node.score:.4f}")

### 刚才发生了什么？

HyDE 的完整流程：

```
用户问题: "如何保证代码质量？"
    ↓
LLM 生成假设性答案: "保证代码质量需要多方面的实践，包括编写单元测试..."
    ↓
用假设性答案的 embedding 去向量索引检索
    ↓
检索到更相关的节点（例如测试策略文档）
    ↓
将检索结果 + 原始问题发给 LLM 生成最终回答
```

`include_original=True` 表示同时使用原始问题和假设性答案的 embedding 检索，提高召回率。

HyDE 特别适合以下场景：
- 用户的问题很抽象（如「如何保证代码质量」），不包含文档中的关键词
- 用户用口语化的方式提问，和文档的书面表达差异大
- 问题和答案在语义空间中距离较远

> **注意**：HyDE 会额外调用一次 LLM（用于生成假设性答案），因此速度更慢、成本更高。适合对检索质量要求较高的场景。

> **与 LangChain 对比**：LangChain 没有内置 HyDE 组件，需要手动实现：先调用 LLM 生成假设文档，再做 embedding 检索。LlamaIndex 的 `HyDEQueryTransform` 一行代码搞定。

## 6. 子问题分解

当用户提出一个复杂问题（涉及多个方面），单次检索往往无法覆盖所有相关内容。`SubQuestionQueryEngine` 能自动将复杂问题**拆解为多个子问题**，分别检索和回答，最后合成完整答案。

In [ ]:
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.query_engine import SubQuestionQueryEngine

# 将索引包装为工具
query_engine_tool = QueryEngineTool(
    query_engine=index.as_query_engine(),
    metadata=ToolMetadata(
        name="programming_knowledge",
        description="编程和软件开发相关的知识库，涵盖 Python、JavaScript、Web 开发、数据库、API 设计、测试和 DevOps",
    ),
)

# 子问题引擎：自动将复杂问题拆解为子问题
sub_question_engine = SubQuestionQueryEngine.from_defaults(
    query_engine_tools=[query_engine_tool],
)

print("SubQuestionQueryEngine 已创建")

In [ ]:
# 提一个涉及多个主题的复杂问题
response = sub_question_engine.query(
    "比较 Python 和 JavaScript 的特点，以及它们各自适合什么场景？"
)
print(response)

### 刚才发生了什么？

`SubQuestionQueryEngine` 的工作流程：

```
用户问题: "比较 Python 和 JavaScript 的特点，以及它们各自适合什么场景？"
    ↓
LLM 拆解为子问题:
  子问题1: "Python 有什么特点？"
  子问题2: "JavaScript 有什么特点？"
  子问题3: "Python 适合什么场景？"
  子问题4: "JavaScript 适合什么场景？"
    ↓
每个子问题分别检索和回答
    ↓
合成最终回答
```

关键概念：

- **QueryEngineTool**：将查询引擎包装为「工具」，附带名称和描述。LLM 会根据描述判断该用哪个工具回答哪个子问题。
- **ToolMetadata**：工具的元信息。`description` 要写清楚知识库的覆盖范围，帮助 LLM 做路由决策。
- 如果有多个知识库（如 Python 知识库、前端知识库），可以传入多个 `QueryEngineTool`，LLM 会自动选择合适的工具回答不同子问题。

> **与 LangChain 对比**：LangChain 没有内置的子问题分解引擎。要实现类似功能，通常需要构建一个 Agent，配合工具调用来完成——逻辑更灵活但代码量更大。LlamaIndex 的 `SubQuestionQueryEngine` 将这个模式封装为开箱即用的组件。

### 常见问题

- **BM25Retriever 报错 `No module named 'llama_index.retrievers.bm25'`**：需要单独安装 `uv add llama-index-retrievers-bm25`。LlamaIndex 的模块化设计意味着非核心组件需要额外安装。
- **混合检索结果和纯向量检索一样**：可能是因为文档太少或查询太泛，两种检索器返回了相同结果。用更大的文档集测试差异更明显。
- **HyDE 生成了不准确的假设性答案**：这是正常的——HyDE 不需要假设答案完全正确，它只需要在语义空间中接近真正的答案即可。
- **SubQuestionQueryEngine 速度慢**：因为它需要多次调用 LLM（拆解问题 + 回答每个子问题 + 合成最终答案），适合离线分析而非实时对话。
- **`SimilarityPostprocessor` 过滤掉了所有节点**：阈值设置太高。建议先打印 `source_nodes` 查看实际分数，再决定合适的 `similarity_cutoff` 值。

## 7. 高级检索策略总览

| 策略 | LlamaIndex | LangChain |
|------|-----------|----------|
| BM25 关键词检索 | `BM25Retriever` 内置 | 需安装 `rank_bm25` 手动集成 |
| 混合检索 | `QueryFusionRetriever` + RRF | `EnsembleRetriever`（需手动配置） |
| 相似度过滤 | `SimilarityPostprocessor` | 需自定义过滤逻辑 |
| 关键词过滤 | `KeywordNodePostprocessor` | 需自定义 |
| HyDE | `HyDEQueryTransform` 一行代码 | 需手动实现 |
| 子问题分解 | `SubQuestionQueryEngine` | 无内置，需用 Agent 实现 |

这张表清晰展示了为什么说**高级检索是 LlamaIndex 的核心竞争力**。这些策略在 LlamaIndex 中都是内置组件，声明式配置、即插即用；而在 LangChain 中大多需要自行实现或组装。

选择建议：

| 场景 | 推荐策略 |
|------|----------|
| 通用文档问答 | 混合检索（向量 + BM25） |
| 用户查询包含专有名词/缩写 | BM25 或混合检索 |
| 用户提问模糊、抽象 | HyDE 查询变换 |
| 复杂多方面问题 | SubQuestionQueryEngine |
| 需要精确控制结果质量 | 后处理器（相似度/关键词过滤） |

## 总结

本章学习了：
- 基础向量检索的局限性（语义匹配 vs 精确匹配）
- BM25 关键词检索
- QueryFusionRetriever 混合检索（向量 + BM25 + RRF 融合）
- 节点后处理器：相似度过滤、关键词过滤
- HyDE 查询变换：用假设性答案改善检索
- SubQuestionQueryEngine 子问题分解
- 对比 LangChain：这些策略在 LlamaIndex 中是内置组件

## 下一章预告

**第六章：对话引擎与多文档 Agent** —— 目前的查询引擎是无状态的，每次提问互不关联。下一章学习如何构建带记忆的对话引擎，以及如何让 Agent 在多个知识库之间自动路由。